 # Detector de queimadas no Parque Nacional da Chapada Diamantina usando dados do FIRMS (Fire Information for Resource Management System) da NASA.

In [1]:
import os
import geopandas as gpd
import pandas as pd
import requests
from io import StringIO
from pathlib import Path
import folium
from dotenv import load_dotenv
import hashlib

In [2]:
def generate_fire_id(row):
    identificador = (
        f"{row['latitude']:.6f}|"
        f"{row['longitude']:.6f}|"
        f"{row['acq_date']}|"
        f"{row['acq_time']}|"
        f"{row['satellite']}"
    )

    hash_id = hashlib.sha256(
        identificador.encode("utf-8")
    ).hexdigest()[:16]

    return f"FIRMS-{hash_id}"

In [3]:
# caminho raiz do projeto
BASE_DIR = Path(Path.cwd()).resolve().parent

# chamar as variaveis de ambiente
load_dotenv()

#Coletar o bounding box da chapada
def get_bbox(kml=BASE_DIR / 'data' / 'shapefile' / 'PARNA_Chap_Diamantina.kml'):

    coord = gpd.read_file(kml)

    sw = [coord.total_bounds[1], coord.total_bounds[0]]
    ne = [coord.total_bounds[3], coord.total_bounds[2]]

    return sw, ne


def get_firms(sw, ne):

    API_KEY = os.getenv("FIRMS_API_KEY")
    SENSORS = ['MODIS_NRT',
               'VIIRS_NOAA20_NRT',
               'VIIRS_NOAA21_NRT',
               'VIIRS_SNPP_NRT']

    bbox = f"{sw[1]},{sw[0]},{ne[1]},{ne[0]}"

    dados = []

    for sensor in SENSORS:
        url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{API_KEY}/{sensor}/{bbox}/5"
        print(f"COLETANDO DADOS DO SATELITE: {sensor}")

        try:
            response = requests.get(url)
            response.raise_for_status()  # Verifica se a requisição foi bem-sucedida

            #verifica se o sensor encontrou algum dado
            if response.text.strip() == "":
                print(f"NENHUM DADO ENCONTRADO NO SATELITE: {sensor}")
                continue

            df = pd.read_csv(StringIO(response.text))
            df['sensor'] = sensor
            dados.append(df)

            print(f"total de dados encontrados: {len(df)}")
        except Exception as e:
            print(f"ERRO AO COLETAR DADOS DO SATELITE {sensor}: {e}")

    if dados:
        df = pd.concat(dados)
        df["fire_id"] = df.apply(generate_fire_id, axis=1)
        return df

In [4]:
# Mapa destacando o bounding box da Chapada Diamantina
sw,ne = get_bbox()

center_lat = (sw[0] + ne[0]) / 2
center_lon = (sw[1] + ne[1]) / 2

m = folium.Map(location=[center_lat, center_lon], zoom_start=9)

folium.Rectangle(
    bounds=[sw, ne],
    color="#ff7800",
    fill=True,
    fill_color="#ffff00",
    fill_opacity=0.2
).add_to(m)

# Display the map
m

Agora vamos coletar os dados de queimadas do FIRMS para o bounding box da Chapada Diamantina}.

In [5]:
df = get_firms(sw, ne)

COLETANDO DADOS DO SATELITE: MODIS_NRT
total de dados encontrados: 0
COLETANDO DADOS DO SATELITE: VIIRS_NOAA20_NRT
total de dados encontrados: 5
COLETANDO DADOS DO SATELITE: VIIRS_NOAA21_NRT
total de dados encontrados: 3
COLETANDO DADOS DO SATELITE: VIIRS_SNPP_NRT
total de dados encontrados: 3


In [6]:
from datetime import datetime

if not df.empty:
    print(f"Total de registros coletados: {len(df)}")
    # criar timestamp
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    # definir caminho do arquivo
    output_file = BASE_DIR / "data" / "fire" / "raw"
    fire_file = output_file / f"queimada_{timestamp}.csv" #arquivo de queimada

    # criar diretório se não existir
    Path(output_file).mkdir(parents=True, exist_ok=True)

    # salva arquivo
    df.to_csv(fire_file, index=False)

else:
    print("Nenhum dado de queimadas encontrado para o período especificado.")

Total de registros coletados: 11


In [7]:
df

,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_t31,frp,daynight,sensor,bright_ti4,bright_ti5,fire_id
0,-13.2512,-41.32436,NaN,0.4,0.37,2026-09-01,1630,N20,VIIRS,n,2.0NRT,NaN,4.29,D,VIIRS_NOAA20_NRT,335.33,307.17,FIRMS-4f37cad35c96e0bf
1,-12.5361,-41.53743,NaN,0.4,0.37,2026-09-01,1630,N20,VIIRS,n,2.0NRT,NaN,1.51,D,VIIRS_NOAA20_NRT,332.97,303.17,FIRMS-c71fab8c4125e790
2,-12.53472,-41.53804,NaN,0.4,0.37,2026-09-01,1630,N20,VIIRS,n,2.0NRT,NaN,1.98,D,VIIRS_NOAA20_NRT,332.95,304.01,FIRMS-01d41d5293df1302
3,-12.46659,-41.31373,NaN,0.57,0.52,2026-09-02,338,N20,VIIRS,n,2.0NRT,NaN,6.25,N,VIIRS_NOAA20_NRT,334.95,289.74,FIRMS-e929ba941f7b586c
4,-12.66647,-41.15562,NaN,0.45,0.39,2026-09-02,1611,N20,VIIRS,n,2.0NRT,NaN,3.56,D,VIIRS_NOAA20_NRT,337.09,310.12,FIRMS-32b27ebad5f6dc86
0,-12.46987,-41.3113,NaN,0.57,0.43,2026-09-02,421,N21,VIIRS,n,2.0NRT,NaN,2.2,N,VIIRS_NOAA21_NRT,309.80,290.20,FIRMS-467aa6d395ade79a
1,-12.46918,-41.31629,NaN,0.57,0.43,2026-09-02,421,N21,VIIRS,n,2.0NRT,NaN,2.2,N,VIIRS_NOAA21_NRT,314.92,290.10,FIRMS-f8a44a0652b1e10e
2,-12.46519,-41.31576,NaN,0.57,0.43,2026-09-02,421,N21,VIIRS,n,2.0NRT,NaN,2.2,N,VIIRS_NOAA21_NRT,306.10,289.70,FIRMS-9261c4666e1987a4
0,-12.46959,-41.3175,NaN,0.75,0.77,2026-09-02,317,N,VIIRS,n,2.0NRT,NaN,1.55,N,VIIRS_SNPP_NRT,311.63,287.41,FIRMS-da58d2ff4bf53956
1,-12.46559,-41.31358,NaN,0.75,0.77,2026-09-02,317,N,VIIRS,n,2.0NRT,NaN,4.01,N,VIIRS_SNPP_NRT,316.44,287.48,FIRMS-e3b2ef8d54e876b8


### Plotando dados de queimadas no mapa

In [8]:
m = folium.Map(location=[center_lat, center_lon], zoom_start=9)
fire_coord = [list(queimada) for queimada in zip(df['latitude'], df['longitude'])]

for idx, row in df.iterrows():
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=f"Data: {row['acq_date']}<br>"
              f"Sensor: {row['sensor']}<br>"
              f"Confiança: {row['confidence']}",
        icon=folium.Icon(color='red', icon='fire', prefix='fa')
    ).add_to(m)
m

In [9]:
def load_boundary(kml):
    """
    Carrega o arquivo KML do Parque Nacional da Chapada Diamantina e retorna um GeoDataFrame.
    """
    return gpd.read_file(kml).union_all()

def filter_by_polygon(df,
                      kml=BASE_DIR / 'data' / 'shapefile' / 'PARNA_Chap_Diamantina.kml'):
    """
    Filtra os dados de queimadas para incluir apenas aqueles dentro do polígono fornecido.
    """

    #Carrega o poligono
    polygons = load_boundary(kml)

    # Converte o df para GeoDataFrame
    gdf = gpd.GeoDataFrame(df,
                           geometry=gpd.points_from_xy(df['longitude'],df['latitude']),
                           crs="EPSG:4326")

    # Filtra os pontos que estão dentro do polígono
    gdf_filtered = gdf.geometry.within(polygons)
    df_filtred = gdf[gdf_filtered].drop(columns='geometry')

    return df_filtred


In [10]:
filter_by_polygon(df=df)

,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_t31,frp,daynight,sensor,bright_ti4,bright_ti5,fire_id


In [11]:
df.columns

Index(['latitude', 'longitude', 'brightness', 'scan', 'track', 'acq_date',
       'acq_time', 'satellite', 'instrument', 'confidence', 'version',
       'bright_t31', 'frp', 'daynight', 'sensor', 'bright_ti4', 'bright_ti5',
       'fire_id'],
      dtype='str')